In [14]:
import os 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')


llm = ChatOpenAI(
    model='gpt-5-nano',
    temperature=0.2,
    api_key=os.getenv('OPENAI_API_KEY')
)

In [15]:
profile = {
    "name": "Sarah",
    "full_name": "Sarah Chen",
    "user_profile_background": "Engenheira de software sênior liderando uma equipe de 5 desenvolvedores",
}

In [16]:
prompt_instructions = {
    "triage_rules": {
        "ignore": "Newsletters de marketing, e-mails de spam, comunicados gerais da empresa",
        "notify": "Membro da equipe doente, notificações do sistema de build, atualizações de status de projeto",
        "respond": "Perguntas diretas de membros da equipe, solicitações de reunião, relatórios de bugs críticos",
    },
    "agent_instructions": "Use estas ferramentas quando apropriado para ajudar a gerenciar as tarefas de Sarah de forma eficiente."
}

In [17]:
email = {
    "from": "Alice Smith <alice.smith@company.com>",
    "to": "Sarah Chen <sarah.chen@company.com>",
    "subject": "Dúvida rápida sobre a documentação da API",
    "body": """
Olá Sarah,

Eu estava revisando a documentação da API para o novo serviço de autenticação e notei que alguns endpoints parecem estar faltando nas especificações. Você poderia me ajudar a esclarecer se isso foi intencional ou se devemos atualizar a documentação?

Especificamente, estou procurando por:
- /auth/refresh
- - /auth/validate

Obrigada!
Alice""",
}

In [18]:
from langchain.chat_models import init_chat_model
from typing_extensions import TypedDict, Literal, Annotated
from pydantic import BaseModel, Field

In [19]:
class Router(BaseModel): 
    """Analisa o e-mail não lido e o roteia de acordo com seu conteúdo."""

    reasoning: str = Field(
        description="Raciocínio passo a passo por trás da classificação."
    )
    classification: Literal["ignore", "respond", "notify"] = Field(
        description="A classificação de um e-mail: 'ignore' para e-mails irrelevantes, "
        "'notify' para informações importantes que não precisam de resposta, "
        "'respond' para e-mails que precisam de uma resposta",
    )

In [20]:
llm_router = llm.with_structured_output(Router)

In [21]:
from prompts import triage_system_prompt, triage_user_prompt, agent_system_prompt


In [22]:
system_prompt = triage_system_prompt.format(
    full_name=profile["full_name"],
    name=profile["name"],
    examples=None,
    user_profile_background=profile["user_profile_background"],
    triage_no=prompt_instructions["triage_rules"]["ignore"],
    triage_notify=prompt_instructions["triage_rules"]["notify"],
    triage_email=prompt_instructions["triage_rules"]["respond"],
)

In [23]:
user_prompt = triage_user_prompt.format(
    author=email["from"],
    to=email["to"],
    subject=email["subject"],
    email_thread=email["body"],
)

In [24]:
result = llm_router.invoke(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
)

In [25]:
print(result)

reasoning="A mensagem é uma pergunta direta de Alice para Sarah sobre a documentação da API, solicitando esclarecimentos se endpoints estão intencionais ou se devem ser atualizados. Isso exige uma resposta direta de Sarah (ou encaminhamento para a pessoa certa) e não é spam/marketing nem apenas uma notificação de status. Portanto, classify como 'respond'. Ações recomendadas: confirmar com a equipe de API/documentação se /auth/refresh e /auth/validate estão ausentes intencionalmente, e, se necessário, preparar uma resposta para Alice com o status atual e próximos passos (ex.: atualizar documentação ou adicionar endpoints)." classification='respond'


In [26]:
from langchain_core.tools import tool
from langchain.agents import create_agent

In [27]:
@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Escreve e envia um e-mail."""
    # Resposta de placeholder - em um aplicativo real, enviaria o e-mail
    return f"E-mail enviado para {to} com o assunto '{subject}'"

In [28]:
@tool
def schedule_meeting(
    attendees: list[str], 
    subject: str, 
    duration_minutes: int, 
    preferred_day: str
) -> str:
    """Agenda uma reunião no calendário."""

    return f"Reunião '{subject}' agendada para {preferred_day} com {len(attendees)} participantes"

In [29]:
@tool
def check_calendar_availability(day: str) -> str:
    """Verifica a disponibilidade do calendário para um determinado dia."""

    return f"Horários disponíveis em {day}: 9:00 AM, 2:00 PM, 4:00 PM"

In [30]:
def create_prompt(state):
    return [
        {
            "role": "system",
            "content": agent_system_prompt.format(
                instructions=prompt_instructions["agent_instructions"],
                **profile
            )
        }
    ] + state['messages']

In [31]:
print(agent_system_prompt)


< Função >
Você é o(a) assistente executivo(a) de {full_name}. Você é um(a) assistente executivo(a) de alto nível que se preocupa com o desempenho de {name} o máximo possível.
</ Função >

< Ferramentas >
Você tem acesso às seguintes ferramentas para ajudar a gerenciar as comunicações e a agenda de {name}:

1. write_email(to, subject, content) - Envia e-mails para destinatários especificados
2. schedule_meeting(attendees, subject, duration_minutes, preferred_day) - Agenda reuniões no calendário
3. check_calendar_availability(day) - Verifica os horários disponíveis em um determinado dia
</ Ferramentas >

< Instruções >
{instructions}
</ Instruções >



In [32]:
tools=[write_email, schedule_meeting, check_calendar_availability]

In [ ]:
agent_prompt = agent_system_prompt.format(
    instructions=prompt_instructions["agent_instructions"],
    **profile,
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=agent_prompt,
)


ValidationError: 2 validation errors for SystemMessage
content.str
  Input should be a valid string [type=string_type, input_value=<function create_prompt at 0x112fc9620>, input_type=function]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
content.list[union[str,dict[any,any]]]
  Input should be a valid list [type=list_type, input_value=<function create_prompt at 0x112fc9620>, input_type=function]
    For further information visit https://errors.pydantic.dev/2.13/v/list_type

In [35]:
response = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "Qual é a minha disponibilidade para terca-feira?"
        }]
    })

NameError: name 'agent' is not defined

In [ ]:
response["message"][-1].pretty_print()